In [36]:
import time
import pandas as pd
import numpy as np
import pennylane as qml
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.svm import SVR

In [37]:
X = pd.read_csv("../dataset/j_kampe.csv")
y = pd.read_csv("../dataset/distances.csv")["distance"]

X = X[1000:2000].values
y = y[1000:2000].values.ravel()

X_train = X[:int(0.8 * X.shape[0])]
X_test  = X[int(0.8 * X.shape[0]):]
y_train = y[:int(0.8 * y.shape[0])]
y_test  = y[int(0.8 * y.shape[0]):]

In [38]:
pca = PCA(n_components=10)
X_train_reduced = pca.fit_transform(X_train)
X_test_reduced = pca.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_reduced)
X_test_scaled = scaler.transform(X_test_reduced)
print(X_train_scaled.shape)

(800, 10)


In [39]:
dev = qml.device("default.qubit", wires=10)

@qml.qnode(dev)
def kernel(x1, x2, n_qubits):
    qml.AngleEmbedding(x1, wires=range(n_qubits))
    qml.adjoint(qml.AngleEmbedding)(x2, wires=range(n_qubits))
    return qml.expval(qml.Projector([0] * n_qubits, wires=range(n_qubits)))

In [40]:
def kernel_mat(A, B):
    mat = []
    for a in A:
        row = []
        for b in B:
            row.append(kernel(a, b, n_qubits=10))
        mat.append(row)
    return np.array(mat)

In [41]:
svr = SVR(kernel=kernel_mat)
svr.fit(X_train_scaled, y_train)
y_pred = svr.predict(X_test_scaled)

rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"RMSE: {rmse}")
print(f"R2: {r2}")

RMSE: 0.2707964536090612
R2: 0.17292691274170668


In [42]:
random_forest_features = pd.read_csv("../results/random_forest_feature_selection.csv")["feature"].tolist()
correlation_features = pd.read_csv("../results/correlation_feature_selection.csv")["feature"].tolist()
gevrey_method_features = pd.read_csv("../results/gevrey_method_feature_selection.csv")["feature"].tolist()
mrmr_10_features = pd.read_csv("../results/mrmr_10_features.csv")["feature"].tolist()

features_map = {}

features_map["Random Forest"] = random_forest_features
features_map["Correlation"] = correlation_features[:10]  # Limiting to top 10 features
features_map["Gevrey Method (10 features)"] = gevrey_method_features[:10]  # Limiting to top 10 features
features_map["mRMR (10 features)"] = mrmr_10_features

In [43]:
class DatasetScalerService:
    MAX_LIMIT = 100_000
    OFFSET    = 1_000

    def __init__(self, features: list[str]):
        self.__scaler_X = StandardScaler()
        self.__X_original = pd.read_csv("../dataset/j_kampe.csv")
        self.__y_original = pd.read_csv("../dataset/distances.csv")["distance"]
        self.__features = features

    def get_scaled_data(self, limit: int = 11_000):
        if limit + self.OFFSET > self.MAX_LIMIT:
            limit = self.MAX_LIMIT

        X = self.__X_original[self.__features].values
        y = self.__y_original.values.ravel()

        X = X[self.OFFSET:limit + self.OFFSET]
        y = y[self.OFFSET:limit + self.OFFSET]

        X_train = X[:int(0.8 * X.shape[0])]
        X_test  = X[int(0.8 * X.shape[0]):]
        y_train = y[:int(0.8 * y.shape[0])]
        y_test  = y[int(0.8 * y.shape[0]):]

        X_train_scaled = self.__scaler_X.fit_transform(X_train)
        X_test_scaled  = self.__scaler_X.transform(X_test)
        return X_train_scaled, X_test_scaled, y_train, y_test

In [44]:
results = []
C = 10
epsilon = 0.001

for feature_name, features in features_map.items():
    n_qubits = len(features)

    print(f"Running QSVR experiment with feature set: {feature_name} ({n_qubits} features)")

    dataset_service = DatasetScalerService(features)
    X_train, X_test, y_train, y_test = dataset_service.get_scaled_data(limit=1_000)

    svr = SVR(kernel=kernel_mat, C=C, epsilon=epsilon)

    start = time.time()
    svr.fit(X_train, y_train)
    y_pred = svr.predict(X_test)
    end = time.time()
    elapsed_time = end - start

    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("RMSE:", rmse)
    print("R2:", r2)
    print("Elapsed time (s):", elapsed_time)
    print("-" * 40)

    results.append({
        "feature_name": feature_name,
        "rmse": rmse,
        "r2": r2,
        "n_qubits": n_qubits,
        "elapsed_time": elapsed_time
    })

results_df = pd.DataFrame(results)
results_df.to_csv("../results/qsvr_v2_experiment_8_angleencoding_results.csv", index=False)
print(results_df)

Running QSVR experiment with feature set: Random Forest (5 features)
RMSE: 0.09017064613195605
R2: 0.908295994028409
Elapsed time (s): 1332.626424074173
----------------------------------------
Running QSVR experiment with feature set: Correlation (10 features)
RMSE: 0.357524665157026
R2: -0.4416836110262068
Elapsed time (s): 1936.0753836631775
----------------------------------------
Running QSVR experiment with feature set: Gevrey Method (10 features) (10 features)
RMSE: 0.12286877151747046
R2: 0.8297288863782236
Elapsed time (s): 1882.3664271831512
----------------------------------------
Running QSVR experiment with feature set: mRMR (10 features) (10 features)
RMSE: 0.15199896562986168
R2: 0.7394211222429066
Elapsed time (s): 1869.2784142494202
----------------------------------------
                  feature_name      rmse        r2  n_qubits  elapsed_time
0                Random Forest  0.090171  0.908296         5   1332.626424
1                  Correlation  0.357525 -0.44168